In [1]:
import csv
import math

csv_path = "scene_faces.csv"

RAW_KEY = "/grscenes_commercial_raw/scenes/"
COMM_KEY = "/grscenes_commercial/scenes/"

def extract_scene_id(scene_path):
    """
    Extract scene ID like:
    MV4AFHQKTKJZ2AABAAAAADY8
    """
    for part in scene_path.split("/"):
        if part.endswith("_usd") and part.startswith("M"):
            return part[:-4]  # remove '_usd'
    return None


raw_faces_total = 0
commercial_faces_total = 0

raw_scene_ids = set()
commercial_faces_by_id = {}

with open(csv_path, newline="") as f:
    reader = csv.reader(f)
    header = next(reader)

    for row in reader:
        if len(row) < 2:
            continue  # skip malformed rows

        scene_path = row[0]
        try:
            total_faces = int(float(row[1]))
        except ValueError:
            continue

        scene_id = extract_scene_id(scene_path)
        if scene_id is None:
            continue

        # --- RAW scenes ---
        if RAW_KEY in scene_path:
            raw_faces_total += total_faces
            raw_scene_ids.add(scene_id)

        # --- COMMERCIAL scenes ---
        elif COMM_KEY in scene_path:
            commercial_faces_by_id[scene_id] = (
                commercial_faces_by_id.get(scene_id, 0) + total_faces
            )

# Sum commercial faces only for scenes that exist in raw
for scene_id in raw_scene_ids:
    commercial_faces_total += commercial_faces_by_id.get(scene_id, 0)

num_scenes = len(raw_scene_ids)

print("Number of raw scenes:", num_scenes)
print("Total faces in grscenes_commercial_raw:", raw_faces_total)
print("Total faces in corresponding grscenes_commercial:", commercial_faces_total)

print(
    "Average faces in grscenes_commercial_raw:",
    math.floor(raw_faces_total / num_scenes)
)
print(
    "Average faces in corresponding grscenes_commercial:",
    math.floor(commercial_faces_total / num_scenes)
)

print(
    "Decimation percentage:",
    1 - (commercial_faces_total / raw_faces_total)
)


Number of raw scenes: 12
Total faces in grscenes_commercial_raw: 261298405
Total faces in corresponding grscenes_commercial: 26372776
Average faces in grscenes_commercial_raw: 21774867
Average faces in corresponding grscenes_commercial: 2197731
Decimation percentage: 0.8990702756107524
